# visloc-rs README hero on a large scene (Google Colab GPU)

Photos → SfM → 3D Gaussian Splatting → mesh with `gsplat_photos`, then the orbiting
hero GIF with `scripts/make_readme_hero.py`, on the **Tanks and Temples Courthouse**
capture (1,106 video frames of a whole town square; every 2nd frame is used by default).

1. `Runtime → Change runtime type → GPU` (T4 is enough; L4/A100 are faster).
2. `Runtime → Run all`. Keep the tab open; the full run took 38 min on an L4 (longer on a T4).
3. The GIF is shown at the end and saved to `/content/out/` (and to Google Drive if
   `USE_DRIVE`). With a `GH_TOKEN` Colab secret and `PUSH = True`, it is committed to
   `BRANCH`.

With `USE_DRIVE = True`, the trained run is kept in Drive, so re-running only redoes
missing steps (e.g. tweak the orbit flags and re-render the GIF in a minute).

In [ ]:
# ---- parameters ----
REPO = 'https://github.com/rsasaki0109/visloc-rs.git'
BRANCH = 'main'
SCENE = 'Courthouse'          # any zip in hf.co/datasets/hongliu6/tanks_and_temples (Barn, Church, Courthouse, ...)
FRAME_STRIDE = 2              # use every Nth frame (Courthouse: 1106 -> 553 photos)
MAX_SIZE = 1280               # long side of the working images
EXHAUSTIVE_MAX = 600          # match every photo pair up to this many photos (closes the loop around the square)
STEPS = 30000                 # 3DGS training steps
NORMAL_WEIGHT = 0.005         # depth-normal loss (smoother mesh), as in the README run
HERO_FLAGS = ('--elev 40 --radius-mult 2.6 --target-height 0.25 --zoom 1.15 --width 640 '
              '--filter-dist-mult 4 --filter-isolated 6 --point-size 3.5')
USE_DRIVE = True              # keep the run in Google Drive (survives a disconnect)
PUSH = False                  # commit the GIF to BRANCH (needs a GH_TOKEN Colab secret)

In [ ]:
import os, subprocess, glob, re, json, shutil, time

def sh(cmd, check=True, quiet=False):
    print('$', cmd, flush=True)
    p = subprocess.Popen(['bash', '-o', 'pipefail', '-c', cmd], text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet:
            print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        if quiet:
            print(''.join(out[-60:]))
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return ''.join(out)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN = f'/content/drive/MyDrive/visloc-hero/{SCENE.lower()}'
else:
    RUN = f'/content/runs/{SCENE.lower()}'
os.makedirs(RUN, exist_ok=True)
os.makedirs('/content/out', exist_ok=True)
print('run dir:', RUN)
sh('nvidia-smi')

## 1. Vulkan for the NVIDIA GPU

visloc-rs runs on the GPU through wgpu (Vulkan on Linux). Colab ships the NVIDIA
compute driver but not always its Vulkan/EGL user-space part, so this cell adds the
part that matches the running kernel driver exactly: already-present libraries first,
then the Ubuntu package of the same version, then the libraries from NVIDIA's own
installer of that version (extracted only; no kernel module is touched).

In [ ]:
sh('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y '
   'vulkan-tools libvulkan1 libegl1 libgl1 libegl-mesa0 libgl1-mesa-dri ffmpeg > /dev/null', quiet=True)

DRV = sh('nvidia-smi --query-gpu=driver_version --format=csv,noheader', quiet=True).strip().splitlines()[0]
MAJOR = DRV.split('.')[0]
LIBDIR = '/usr/lib/x86_64-linux-gnu'
print('NVIDIA driver', DRV)

def vulkan_gpu():
    r = subprocess.run('vulkaninfo --summary', shell=True, text=True, capture_output=True)
    names = re.findall(r'deviceName\s*=\s*(.+)', r.stdout)
    hw = [n for n in names if 'llvmpipe' not in n and 'SwiftShader' not in n]
    return hw[0].strip() if hw else None

def write_icds(glx='libGLX_nvidia.so.0', egl='libEGL_nvidia.so.0'):
    os.makedirs('/usr/share/vulkan/icd.d', exist_ok=True)
    json.dump({'file_format_version': '1.0.0',
               'ICD': {'library_path': glx, 'api_version': '1.3.242'}},
              open('/usr/share/vulkan/icd.d/nvidia_icd.json', 'w'))
    os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
    json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': egl}},
              open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w'))

def try_present():
    for d in ['/usr/lib64-nvidia', LIBDIR, '/usr/local/nvidia/lib64']:
        hits = sorted(glob.glob(f'{d}/libGLX_nvidia.so.{DRV}'))
        if hits:
            egl = glob.glob(f'{d}/libEGL_nvidia.so.{DRV}')
            write_icds(hits[0], egl[0] if egl else 'libEGL_nvidia.so.0')
            return True
    return False

def try_apt():
    out = subprocess.run(f'apt-cache madison libnvidia-gl-{MAJOR}', shell=True, text=True,
                         capture_output=True).stdout
    vers = [l.split('|')[1].strip() for l in out.splitlines() if '|' in l]
    match = [v for v in vers if v.startswith(DRV)]
    print('apt libnvidia-gl versions:', vers[:5], '-> match', match[:1])
    if not match:
        return False
    sh(f'DEBIAN_FRONTEND=noninteractive apt-get -qq install -y --no-install-recommends '
       f'libnvidia-gl-{MAJOR}={match[0]} > /dev/null', check=False, quiet=True)
    write_icds()
    return True

def try_runfile():
    run = f'/content/NVIDIA-Linux-x86_64-{DRV}.run'
    if not os.path.exists(run):
        for url in [f'https://us.download.nvidia.com/tesla/{DRV}/NVIDIA-Linux-x86_64-{DRV}.run',
                    f'https://us.download.nvidia.com/XFree86/Linux-x86_64/{DRV}/NVIDIA-Linux-x86_64-{DRV}.run']:
            if subprocess.run(f'wget -q -O {run} {url}', shell=True).returncode == 0:
                break
            if os.path.exists(run):
                os.remove(run)
        else:
            return False
    ex = f'/content/nvidia-{DRV}'
    if not os.path.isdir(ex):
        sh(f'sh {run} --extract-only --target {ex} > /dev/null', quiet=True)
    names = ['libGLX_nvidia', 'libnvidia-glcore', 'libnvidia-glvkspirv', 'libnvidia-tls',
             'libnvidia-gpucomp', 'libEGL_nvidia', 'libnvidia-eglcore', 'libnvidia-glsi']
    for n in names:
        src = f'{ex}/{n}.so.{DRV}'
        if os.path.exists(src):
            shutil.copy(src, LIBDIR)
            if n in ('libGLX_nvidia', 'libEGL_nvidia'):
                link = f'{LIBDIR}/{n}.so.0'
                if os.path.lexists(link):
                    os.remove(link)
                os.symlink(f'{n}.so.{DRV}', link)
    sh('ldconfig', quiet=True)
    write_icds()
    return True

gpu = vulkan_gpu()
for name, step in [('present', try_present), ('apt', try_apt), ('runfile', try_runfile)]:
    if gpu:
        break
    print(f'-- trying {name}')
    if step():
        gpu = vulkan_gpu()
print('Vulkan GPU:', gpu)
assert gpu, 'no hardware Vulkan device; see `vulkaninfo --summary` above'
os.environ['WGPU_BACKEND'] = 'vulkan'

## 2. Build visloc-rs

In [ ]:
if not os.path.exists('/root/.cargo/bin/cargo'):
    sh('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null', quiet=True)
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
if not os.path.isdir('/content/visloc-rs'):
    sh(f'git clone --depth 1 -b {BRANCH} {REPO} /content/visloc-rs')
else:
    sh(f'cd /content/visloc-rs && git pull --ff-only origin {BRANCH}')
t = time.time()
sh('cd /content/visloc-rs && cargo build --release -p visloc-gsplat-train --features gpu,euroc '
   '--example gsplat_photos --example gsplat_eval 2>&1 | tail -3')
print(f'build {time.time() - t:.0f}s')
BIN = '/content/visloc-rs/target/release/examples'

## 3. Download the photos

In [ ]:
IMAGES = f'/content/data/{SCENE}_stride{FRAME_STRIDE}'
if not os.path.isdir(IMAGES) or not os.listdir(IMAGES):
    z = f'/content/data/{SCENE}.zip'
    os.makedirs('/content/data', exist_ok=True)
    if not os.path.exists(z):
        sh(f'wget -q -O {z} "https://huggingface.co/datasets/hongliu6/tanks_and_temples/resolve/main/{SCENE}.zip"')
    sh(f'cd /content/data && rm -rf {SCENE} && unzip -q {z}')
    frames = sorted(glob.glob(f'/content/data/{SCENE}/*.jpg'))
    os.makedirs(IMAGES, exist_ok=True)
    for f in frames[::FRAME_STRIDE]:
        shutil.move(f, IMAGES)
    shutil.rmtree(f'/content/data/{SCENE}')
N_INPUT = len(os.listdir(IMAGES))
print(N_INPUT, 'photos in', IMAGES)

## 4. Photos → SfM → 3D Gaussian Splatting → mesh

One `gsplat_photos` command. Its log (registration, focal refinement, held-out PSNR,
timings) is saved as `gsplat_photos.log` next to the run. Skipped if the run already
has a `mesh.ply`.

In [ ]:
if not os.path.exists(f'{RUN}/mesh.ply'):
    t = time.time()
    sh(f'{BIN}/gsplat_photos --images {IMAGES} --out {RUN} --max-size {MAX_SIZE} '
       f'--steps {STEPS} --normal-weight {NORMAL_WEIGHT} --exhaustive-max {EXHAUSTIVE_MAX} 2>&1 '
       f'| grep --line-buffered -v -E "^(BA_|INIT_PAIR|BA global|BA local|TIMING|REGISTER)" | tee {RUN}/gsplat_photos.log')
    print(f'gsplat_photos {time.time() - t:.0f}s')
sh(f'grep -E "registered|psnr|wrote|total|triangles|mesh" {RUN}/gsplat_photos.log | tail -12', check=False)

## 5. The hero GIF

Orbit settings are flags of `scripts/make_readme_hero.py` (`HERO_FLAGS` above:
`--elev`, `--radius-mult`, `--zoom`, `--target-height`, `--phase0`, `--point-size`).
Re-running this cell alone re-renders in a few minutes.

In [ ]:
sh('pip -q install moderngl plyfile scipy pillow', quiet=True)
label = f'Tanks and Temples {SCENE}'
sh(f'cd /content/visloc-rs && python scripts/make_readme_hero.py --src {RUN} --work /content/hero '
   f'--gsplat-eval {BIN}/gsplat_eval --scene-label "{label}" --input-images {N_INPUT} '
   f'{HERO_FLAGS} --out /content/out/hero_reconstruction.gif')
if USE_DRIVE:
    shutil.copy('/content/out/hero_reconstruction.gif', RUN)
from IPython.display import Image as IPImage, display
display(IPImage(filename='/content/out/hero_reconstruction.gif'))

## 6. (Optional) commit the GIF

Add a GitHub token with `contents: write` on the repo as a Colab secret named
`GH_TOKEN` (key icon in the left bar), set `PUSH = True`, and run this cell. The commit
message carries the run summary from `gsplat_photos.log`. Otherwise download
`/content/out/hero_reconstruction.gif` and `gsplat_photos.log` from the Files pane.

In [ ]:
shutil.copy(f'{RUN}/gsplat_photos.log', '/content/out/gsplat_photos.log')
if PUSH:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    summary = sh(f'grep -E "registered|psnr|total|triangles" {RUN}/gsplat_photos.log | tail -8',
                 check=False, quiet=True)
    gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader', quiet=True).strip()
    msg = (f'docs(readme): hero GIF on Tanks and Temples {SCENE}\n\n'
           f'{N_INPUT} photos (every {FRAME_STRIDE}th frame), --max-size {MAX_SIZE}, '
           f'--steps {STEPS}, hero flags: {HERO_FLAGS}; run on Colab {gpu_name}.\n\n{summary}')
    open('/content/commit_msg.txt', 'w').write(msg)
    repo = '/content/visloc-rs'
    shutil.copy('/content/out/hero_reconstruction.gif', f'{repo}/docs/assets/hero_reconstruction.gif')
    os.makedirs(f'{repo}/docs/media/hero', exist_ok=True)
    shutil.copy(f'{RUN}/gsplat_photos.log', f'{repo}/docs/media/hero/{SCENE.lower()}_gsplat_photos.log')
    auth = REPO.replace('https://', f'https://x-access-token:{token}@')
    sh(f'cd {repo} && git -c user.name="visloc-rs colab" -c user.email="colab@users.noreply.github.com" '
       f'add docs/assets/hero_reconstruction.gif docs/media/hero && '
       f'git -c user.name="visloc-rs colab" -c user.email="colab@users.noreply.github.com" '
       f'commit -q -F /content/commit_msg.txt && git push -q {auth} HEAD:{BRANCH}')
    print('pushed to', BRANCH)